# Interacting with Uniswap using Universal Router
In last week, we interacted with Uniswap using their GUI. However, GUI is not such reliable and efficient way to interact with Uniswap. We found that there are some issues when we try to swap tokens using Uniswap GUI. For example, we cannot swap some tokens because the API is not working properly. 

<img src="./figs/uniswap_fig1.png" width="50%"/>
<img src="./figs/uniswap_fig2.png" width="40%"/>


This week, we will interact with Uniswap using code.
We understand how to build transactions and approve the right smart contracts to use our funds to perform these swaps. 

Then, we execute an atomic swap via Code.

### ERC-2612 vs Permit2
- `ERC-2612` is a token standard (technical specification)
- `Permit2` is a specific smart contract (infrastructure) developed by Uniswap.
- Both aim to solve the same problem: the cumbersome and expensive Token approval mechanism.

### ERC-2612: Gasless Approval via Signatures
To understand ERC-2612, we first look at how the **traditional ERC-20** works.

#### Traditional Pain Point (ERC-20)
If you want to deposit USDT into Uniswap for trading, you must complete **two separate transactions**:
1. **Approve**: Send a transaction to the USDT contract to authorize: “I allow Uniswap to spend my 100 USDT.”
   This **requires Gas** and waits for on-chain confirmation.
2. **Transfer/Execute**: Send another transaction to Uniswap to start trading. Only then can Uniswap transfer your tokens.
   This **costs Gas again**.

**Drawback**: Poor user experience (two waits) and high Gas fees.

#### ERC-2612: Signature-Based Approval
ERC-2612 extends ERC-20 by adding a `permit` function.
1. **Sign (off-chain)**: Instead of sending an on-chain transaction, you simply **sign a message** in your wallet (free, off-chain).
   The signature states: “I authorize Uniswap to spend 100 USDT until tomorrow.”
2. **Execute**: You send the signature to Uniswap.
   During the swap, Uniswap uses your signature to call the token’s `permit` function (instant approval), then completes the transfer.

**Advantage**: Two steps merged into one, saves Gas, and improves UX.

In summary:
It allows tokens to support **approval via signature**, rather than requiring an on-chain transaction.


## Populate all relavant variables

In [ ]:
# Import web3 library
from web3 import Web3

# Setup variables
infura_key = ''  # Replace with your Infura Project ID or set in .env
wallet_public_address = ''  # Replace with your wallet address or set in .env
wallet_private_key = ''  # Replace with your private key or set in .env


w3 = Web3(Web3.HTTPProvider(f'https://sepolia.infura.io/v3/{infura_key}'))


print(w3.__dict__)
print(w3.eth.get_block_number())

wallet_public_address = Web3.to_checksum_address(wallet_public_address)

# Define token addresses (USTUSD, USTETH), where USTUSD is the wrapped Ether token
# WETH_token_address = Web3.to_checksum_address('0xfFf9976782d46CC05630D1f6eBAb18b2324d6B14')
USTUSD_token_address = Web3.to_checksum_address('0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c')
USTETH_token_address = Web3.to_checksum_address('0x4E22e9951770b98d4f3B2BA6647f0f40A15A4057')


# USTUSD token contract ABI (ERC-20 standard): https://sepolia.etherscan.io/address/0xfFf9976782d46CC05630D1f6eBAb18b2324d6B14#code 
abi_USTUSD = '[{"constant":true,"inputs":[],"name":"name","outputs":[{"name":"","type":"string"}],"payable":false,"stateMutability":"view","type":"function"},{"constant":false,"inputs":[{"name":"guy","type":"address"},{"name":"wad","type":"uint256"}],"name":"approve","outputs":[{"name":"","type":"bool"}],"payable":false,"stateMutability":"nonpayable","type":"function"},{"constant":true,"inputs":[],"name":"totalSupply","outputs":[{"name":"","type":"uint256"}],"payable":false,"stateMutability":"view","type":"function"},{"constant":false,"inputs":[{"name":"src","type":"address"},{"name":"dst","type":"address"},{"name":"wad","type":"uint256"}],"name":"transferFrom","outputs":[{"name":"","type":"bool"}],"payable":false,"stateMutability":"nonpayable","type":"function"},{"constant":false,"inputs":[{"name":"wad","type":"uint256"}],"name":"withdraw","outputs":[],"payable":false,"stateMutability":"nonpayable","type":"function"},{"constant":true,"inputs":[],"name":"decimals","outputs":[{"name":"","type":"uint8"}],"payable":false,"stateMutability":"view","type":"function"},{"constant":true,"inputs":[{"name":"","type":"address"}],"name":"balanceOf","outputs":[{"name":"","type":"uint256"}],"payable":false,"stateMutability":"view","type":"function"},{"constant":true,"inputs":[],"name":"symbol","outputs":[{"name":"","type":"string"}],"payable":false,"stateMutability":"view","type":"function"},{"constant":false,"inputs":[{"name":"dst","type":"address"},{"name":"wad","type":"uint256"}],"name":"transfer","outputs":[{"name":"","type":"bool"}],"payable":false,"stateMutability":"nonpayable","type":"function"},{"constant":false,"inputs":[],"name":"deposit","outputs":[],"payable":true,"stateMutability":"payable","type":"function"},{"constant":true,"inputs":[{"name":"","type":"address"},{"name":"","type":"address"}],"name":"allowance","outputs":[{"name":"","type":"uint256"}],"payable":false,"stateMutability":"view","type":"function"},{"payable":true,"stateMutability":"payable","type":"fallback"},{"anonymous":false,"inputs":[{"indexed":true,"name":"src","type":"address"},{"indexed":true,"name":"guy","type":"address"},{"indexed":false,"name":"wad","type":"uint256"}],"name":"Approval","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"name":"src","type":"address"},{"indexed":true,"name":"dst","type":"address"},{"indexed":false,"name":"wad","type":"uint256"}],"name":"Transfer","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"name":"dst","type":"address"},{"indexed":false,"name":"wad","type":"uint256"}],"name":"Deposit","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"name":"src","type":"address"},{"indexed":false,"name":"wad","type":"uint256"}],"name":"Withdrawal","type":"event"}]'

# USTETH token contract ABI (ERC-20 standard): https://sepolia.etherscan.io/address/0x4E22e9951770b98d4f3B2BA6647f0f40A15A4057#code
abi_USTETH = '[{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"allowance","type":"uint256"},{"internalType":"uint256","name":"needed","type":"uint256"}],"name":"ERC20InsufficientAllowance","type":"error"},{"inputs":[{"internalType":"address","name":"sender","type":"address"},{"internalType":"uint256","name":"balance","type":"uint256"},{"internalType":"uint256","name":"needed","type":"uint256"}],"name":"ERC20InsufficientBalance","type":"error"},{"inputs":[{"internalType":"address","name":"approver","type":"address"}],"name":"ERC20InvalidApprover","type":"error"},{"inputs":[{"internalType":"address","name":"receiver","type":"address"}],"name":"ERC20InvalidReceiver","type":"error"},{"inputs":[{"internalType":"address","name":"sender","type":"address"}],"name":"ERC20InvalidSender","type":"error"},{"inputs":[{"internalType":"address","name":"spender","type":"address"}],"name":"ERC20InvalidSpender","type":"error"},{"inputs":[{"internalType":"address","name":"owner","type":"address"}],"name":"OwnableInvalidOwner","type":"error"},{"inputs":[{"internalType":"address","name":"account","type":"address"}],"name":"OwnableUnauthorizedAccount","type":"error"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"address","name":"spender","type":"address"},{"indexed":false,"internalType":"uint256","name":"value","type":"uint256"}],"name":"Approval","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"previousOwner","type":"address"},{"indexed":true,"internalType":"address","name":"newOwner","type":"address"}],"name":"OwnershipTransferred","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"from","type":"address"},{"indexed":true,"internalType":"address","name":"to","type":"address"},{"indexed":false,"internalType":"uint256","name":"value","type":"uint256"}],"name":"Transfer","type":"event"},{"inputs":[{"internalType":"address","name":"owner","type":"address"},{"internalType":"address","name":"spender","type":"address"}],"name":"allowance","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"approve","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"account","type":"address"}],"name":"balanceOf","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"decimals","outputs":[{"internalType":"uint8","name":"","type":"uint8"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"name":"mint","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"name","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"owner","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"renounceOwnership","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"symbol","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"totalSupply","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"transfer","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"from","type":"address"},{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"transferFrom","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"newOwner","type":"address"}],"name":"transferOwnership","outputs":[],"stateMutability":"nonpayable","type":"function"}]'

USTUSD_contract = w3.eth.contract(address=USTUSD_token_address, abi=abi_USTUSD)
USTETH_contract = w3.eth.contract(address=USTETH_token_address, abi=abi_USTETH)

print(USTUSD_contract.functions.name().call())
print(USTETH_contract.functions.name().call())

### Universal Router
- Help us to route our swaps (e.g., USDT -> USDC -> ETH) in a single transaction with the best price.
- It is the intermediary that orchestrates how funds flow, handling multi-hop trades, slippage protection, deadline checks, etc.
- It supports both ERC-2612 and Permit2, allowing users to choose their preferred approval method.


In [ ]:
# Define Universal Router address and ABI: https://sepolia.etherscan.io/address/0x3fC91A3afd70395Cd496C647d5a6CC9D4B2b7FAD#code
universal_router_address = Web3.to_checksum_address('0x3fC91A3afd70395Cd496C647d5a6CC9D4B2b7FAD')

abi_universal_router = '[{"inputs":[{"components":[{"internalType":"address","name":"permit2","type":"address"},{"internalType":"address","name":"USTUSD9","type":"address"},{"internalType":"address","name":"seaportV1_5","type":"address"},{"internalType":"address","name":"seaportV1_4","type":"address"},{"internalType":"address","name":"openseaConduit","type":"address"},{"internalType":"address","name":"nftxZap","type":"address"},{"internalType":"address","name":"x2y2","type":"address"},{"internalType":"address","name":"foundation","type":"address"},{"internalType":"address","name":"sudoswap","type":"address"},{"internalType":"address","name":"elementMarket","type":"address"},{"internalType":"address","name":"nft20Zap","type":"address"},{"internalType":"address","name":"cryptopunks","type":"address"},{"internalType":"address","name":"looksRareV2","type":"address"},{"internalType":"address","name":"routerRewardsDistributor","type":"address"},{"internalType":"address","name":"looksRareRewardsDistributor","type":"address"},{"internalType":"address","name":"looksRareToken","type":"address"},{"internalType":"address","name":"v2Factory","type":"address"},{"internalType":"address","name":"v3Factory","type":"address"},{"internalType":"bytes32","name":"pairInitCodeHash","type":"bytes32"},{"internalType":"bytes32","name":"poolInitCodeHash","type":"bytes32"}],"internalType":"struct RouterParameters","name":"params","type":"tuple"}],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[],"name":"BalanceTooLow","type":"error"},{"inputs":[],"name":"BuyPunkFailed","type":"error"},{"inputs":[],"name":"ContractLocked","type":"error"},{"inputs":[],"name":"ETHNotAccepted","type":"error"},{"inputs":[{"internalType":"uint256","name":"commandIndex","type":"uint256"},{"internalType":"bytes","name":"message","type":"bytes"}],"name":"ExecutionFailed","type":"error"},{"inputs":[],"name":"FromAddressIsNotOwner","type":"error"},{"inputs":[],"name":"InsufficientETH","type":"error"},{"inputs":[],"name":"InsufficientToken","type":"error"},{"inputs":[],"name":"InvalidBips","type":"error"},{"inputs":[{"internalType":"uint256","name":"commandType","type":"uint256"}],"name":"InvalidCommandType","type":"error"},{"inputs":[],"name":"InvalidOwnerERC1155","type":"error"},{"inputs":[],"name":"InvalidOwnerERC721","type":"error"},{"inputs":[],"name":"InvalidPath","type":"error"},{"inputs":[],"name":"InvalidReserves","type":"error"},{"inputs":[],"name":"InvalidSpender","type":"error"},{"inputs":[],"name":"LengthMismatch","type":"error"},{"inputs":[],"name":"SliceOutOfBounds","type":"error"},{"inputs":[],"name":"TransactionDeadlinePassed","type":"error"},{"inputs":[],"name":"UnableToClaim","type":"error"},{"inputs":[],"name":"UnsafeCast","type":"error"},{"inputs":[],"name":"V2InvalidPath","type":"error"},{"inputs":[],"name":"V2TooLittleReceived","type":"error"},{"inputs":[],"name":"V2TooMuchRequested","type":"error"},{"inputs":[],"name":"V3InvalidAmountOut","type":"error"},{"inputs":[],"name":"V3InvalidCaller","type":"error"},{"inputs":[],"name":"V3InvalidSwap","type":"error"},{"inputs":[],"name":"V3TooLittleReceived","type":"error"},{"inputs":[],"name":"V3TooMuchRequested","type":"error"},{"anonymous":false,"inputs":[{"indexed":false,"internalType":"uint256","name":"amount","type":"uint256"}],"name":"RewardsSent","type":"event"},{"inputs":[{"internalType":"bytes","name":"looksRareClaim","type":"bytes"}],"name":"collectRewards","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"bytes","name":"commands","type":"bytes"},{"internalType":"bytes[]","name":"inputs","type":"bytes[]"}],"name":"execute","outputs":[],"stateMutability":"payable","type":"function"},{"inputs":[{"internalType":"bytes","name":"commands","type":"bytes"},{"internalType":"bytes[]","name":"inputs","type":"bytes[]"},{"internalType":"uint256","name":"deadline","type":"uint256"}],"name":"execute","outputs":[],"stateMutability":"payable","type":"function"},{"inputs":[{"internalType":"address","name":"","type":"address"},{"internalType":"address","name":"","type":"address"},{"internalType":"uint256[]","name":"","type":"uint256[]"},{"internalType":"uint256[]","name":"","type":"uint256[]"},{"internalType":"bytes","name":"","type":"bytes"}],"name":"onERC1155BatchReceived","outputs":[{"internalType":"bytes4","name":"","type":"bytes4"}],"stateMutability":"pure","type":"function"},{"inputs":[{"internalType":"address","name":"","type":"address"},{"internalType":"address","name":"","type":"address"},{"internalType":"uint256","name":"","type":"uint256"},{"internalType":"uint256","name":"","type":"uint256"},{"internalType":"bytes","name":"","type":"bytes"}],"name":"onERC1155Received","outputs":[{"internalType":"bytes4","name":"","type":"bytes4"}],"stateMutability":"pure","type":"function"},{"inputs":[{"internalType":"address","name":"","type":"address"},{"internalType":"address","name":"","type":"address"},{"internalType":"uint256","name":"","type":"uint256"},{"internalType":"bytes","name":"","type":"bytes"}],"name":"onERC721Received","outputs":[{"internalType":"bytes4","name":"","type":"bytes4"}],"stateMutability":"pure","type":"function"},{"inputs":[{"internalType":"bytes4","name":"interfaceId","type":"bytes4"}],"name":"supportsInterface","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"pure","type":"function"},{"inputs":[{"internalType":"int256","name":"amount0Delta","type":"int256"},{"internalType":"int256","name":"amount1Delta","type":"int256"},{"internalType":"bytes","name":"data","type":"bytes"}],"name":"uniswapV3SwapCallback","outputs":[],"stateMutability":"nonpayable","type":"function"},{"stateMutability":"payable","type":"receive"}]'

# Define Permit2 address and ABI: https://sepolia.etherscan.io/address/0x000000000022D473030F116dDEE9F6B43aC78BA3#code
permit2_address = w3.to_checksum_address('0x000000000022D473030F116dDEE9F6B43aC78BA3')
abi_permit2 = '[{"inputs":[{"internalType":"uint256","name":"deadline","type":"uint256"}],"name":"AllowanceExpired","type":"error"},{"inputs":[],"name":"ExcessiveInvalidation","type":"error"},{"inputs":[{"internalType":"uint256","name":"amount","type":"uint256"}],"name":"InsufficientAllowance","type":"error"},{"inputs":[{"internalType":"uint256","name":"maxAmount","type":"uint256"}],"name":"InvalidAmount","type":"error"},{"inputs":[],"name":"InvalidContractSignature","type":"error"},{"inputs":[],"name":"InvalidNonce","type":"error"},{"inputs":[],"name":"InvalidSignature","type":"error"},{"inputs":[],"name":"InvalidSignatureLength","type":"error"},{"inputs":[],"name":"InvalidSigner","type":"error"},{"inputs":[],"name":"LengthMismatch","type":"error"},{"inputs":[{"internalType":"uint256","name":"signatureDeadline","type":"uint256"}],"name":"SignatureExpired","type":"error"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"address","name":"token","type":"address"},{"indexed":true,"internalType":"address","name":"spender","type":"address"},{"indexed":false,"internalType":"uint160","name":"amount","type":"uint160"},{"indexed":false,"internalType":"uint48","name":"expiration","type":"uint48"}],"name":"Approval","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":false,"internalType":"address","name":"token","type":"address"},{"indexed":false,"internalType":"address","name":"spender","type":"address"}],"name":"Lockdown","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"address","name":"token","type":"address"},{"indexed":true,"internalType":"address","name":"spender","type":"address"},{"indexed":false,"internalType":"uint48","name":"newNonce","type":"uint48"},{"indexed":false,"internalType":"uint48","name":"oldNonce","type":"uint48"}],"name":"NonceInvalidation","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":true,"internalType":"address","name":"token","type":"address"},{"indexed":true,"internalType":"address","name":"spender","type":"address"},{"indexed":false,"internalType":"uint160","name":"amount","type":"uint160"},{"indexed":false,"internalType":"uint48","name":"expiration","type":"uint48"},{"indexed":false,"internalType":"uint48","name":"nonce","type":"uint48"}],"name":"Permit","type":"event"},{"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"owner","type":"address"},{"indexed":false,"internalType":"uint256","name":"word","type":"uint256"},{"indexed":false,"internalType":"uint256","name":"mask","type":"uint256"}],"name":"UnorderedNonceInvalidation","type":"event"},{"inputs":[],"name":"DOMAIN_SEPARATOR","outputs":[{"internalType":"bytes32","name":"","type":"bytes32"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"","type":"address"},{"internalType":"address","name":"","type":"address"},{"internalType":"address","name":"","type":"address"}],"name":"allowance","outputs":[{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"uint48","name":"expiration","type":"uint48"},{"internalType":"uint48","name":"nonce","type":"uint48"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"token","type":"address"},{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"uint48","name":"expiration","type":"uint48"}],"name":"approve","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"token","type":"address"},{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint48","name":"newNonce","type":"uint48"}],"name":"invalidateNonces","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"uint256","name":"wordPos","type":"uint256"},{"internalType":"uint256","name":"mask","type":"uint256"}],"name":"invalidateUnorderedNonces","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"address","name":"spender","type":"address"}],"internalType":"struct IAllowanceTransfer.TokenSpenderPair[]","name":"approvals","type":"tuple[]"}],"name":"lockdown","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"","type":"address"},{"internalType":"uint256","name":"","type":"uint256"}],"name":"nonceBitmap","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"address","name":"owner","type":"address"},{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"uint48","name":"expiration","type":"uint48"},{"internalType":"uint48","name":"nonce","type":"uint48"}],"internalType":"struct IAllowanceTransfer.PermitDetails[]","name":"details","type":"tuple[]"},{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"sigDeadline","type":"uint256"}],"internalType":"struct IAllowanceTransfer.PermitBatch","name":"permitBatch","type":"tuple"},{"internalType":"bytes","name":"signature","type":"bytes"}],"name":"permit","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"owner","type":"address"},{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"uint48","name":"expiration","type":"uint48"},{"internalType":"uint48","name":"nonce","type":"uint48"}],"internalType":"struct IAllowanceTransfer.PermitDetails","name":"details","type":"tuple"},{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"sigDeadline","type":"uint256"}],"internalType":"struct IAllowanceTransfer.PermitSingle","name":"permitSingle","type":"tuple"},{"internalType":"bytes","name":"signature","type":"bytes"}],"name":"permit","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"internalType":"struct ISignatureTransfer.TokenPermissions","name":"permitted","type":"tuple"},{"internalType":"uint256","name":"nonce","type":"uint256"},{"internalType":"uint256","name":"deadline","type":"uint256"}],"internalType":"struct ISignatureTransfer.PermitTransferFrom","name":"permit","type":"tuple"},{"components":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"requestedAmount","type":"uint256"}],"internalType":"struct ISignatureTransfer.SignatureTransferDetails","name":"transferDetails","type":"tuple"},{"internalType":"address","name":"owner","type":"address"},{"internalType":"bytes","name":"signature","type":"bytes"}],"name":"permitTransferFrom","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"internalType":"struct ISignatureTransfer.TokenPermissions[]","name":"permitted","type":"tuple[]"},{"internalType":"uint256","name":"nonce","type":"uint256"},{"internalType":"uint256","name":"deadline","type":"uint256"}],"internalType":"struct ISignatureTransfer.PermitBatchTransferFrom","name":"permit","type":"tuple"},{"components":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"requestedAmount","type":"uint256"}],"internalType":"struct ISignatureTransfer.SignatureTransferDetails[]","name":"transferDetails","type":"tuple[]"},{"internalType":"address","name":"owner","type":"address"},{"internalType":"bytes","name":"signature","type":"bytes"}],"name":"permitTransferFrom","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"internalType":"struct ISignatureTransfer.TokenPermissions","name":"permitted","type":"tuple"},{"internalType":"uint256","name":"nonce","type":"uint256"},{"internalType":"uint256","name":"deadline","type":"uint256"}],"internalType":"struct ISignatureTransfer.PermitTransferFrom","name":"permit","type":"tuple"},{"components":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"requestedAmount","type":"uint256"}],"internalType":"struct ISignatureTransfer.SignatureTransferDetails","name":"transferDetails","type":"tuple"},{"internalType":"address","name":"owner","type":"address"},{"internalType":"bytes32","name":"witness","type":"bytes32"},{"internalType":"string","name":"witnessTypeString","type":"string"},{"internalType":"bytes","name":"signature","type":"bytes"}],"name":"permitWitnessTransferFrom","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"}],"internalType":"struct ISignatureTransfer.TokenPermissions[]","name":"permitted","type":"tuple[]"},{"internalType":"uint256","name":"nonce","type":"uint256"},{"internalType":"uint256","name":"deadline","type":"uint256"}],"internalType":"struct ISignatureTransfer.PermitBatchTransferFrom","name":"permit","type":"tuple"},{"components":[{"internalType":"address","name":"to","type":"address"},{"internalType":"uint256","name":"requestedAmount","type":"uint256"}],"internalType":"struct ISignatureTransfer.SignatureTransferDetails[]","name":"transferDetails","type":"tuple[]"},{"internalType":"address","name":"owner","type":"address"},{"internalType":"bytes32","name":"witness","type":"bytes32"},{"internalType":"string","name":"witnessTypeString","type":"string"},{"internalType":"bytes","name":"signature","type":"bytes"}],"name":"permitWitnessTransferFrom","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"internalType":"address","name":"from","type":"address"},{"internalType":"address","name":"to","type":"address"},{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"address","name":"token","type":"address"}],"internalType":"struct IAllowanceTransfer.AllowanceTransferDetails[]","name":"transferDetails","type":"tuple[]"}],"name":"transferFrom","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"internalType":"address","name":"from","type":"address"},{"internalType":"address","name":"to","type":"address"},{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"address","name":"token","type":"address"}],"name":"transferFrom","outputs":[],"stateMutability":"nonpayable","type":"function"}]'

permit2_contract = w3.eth.contract(address=permit2_address, abi=abi_permit2)
universal_router_contract = w3.eth.contract(address=universal_router_address, abi=abi_universal_router)

# Test calls to Universal Router and Permit2 contracts
print((permit2_contract.functions.DOMAIN_SEPARATOR().call()).hex()) # print the DOMAIN_SEPARATOR of Permit2 contract, which is a hash including unique information about the contract and the chain, used in EIP-712 signatures
print(universal_router_contract.functions.supportsInterface('0x01ffc9a7').call())  # ERC165 interface ID

## Approving USTETH to the Permit2 contract

This step is similar to the ERC-20 `approve` function, but instead of approving Uniswap, we approve the Permit2 contract to spend our USTETH. This is necessary because when we execute the swap, the Universal Router will call Permit2 to transfer our USTETH.

In [ ]:
# approve Permit2 to USTETH with max allowance

permit2_allowance = 2**256 - 1  # max

contract_function = USTETH_contract.functions.approve(
        permit2_address,
        permit2_allowance
)

trx_params = contract_function.build_transaction(
        {
            "from": wallet_public_address,
            "gas": 500_000,
            "maxPriorityFeePerGas": w3.eth.max_priority_fee,
            "maxFeePerGas": 100 * 10**9,
            "type": '0x2',
            "chainId": 11155111,
            "value": 0,
            "nonce": w3.eth.get_transaction_count(wallet_public_address),
        }
    )

signed_txn = w3.eth.account.sign_transaction(trx_params, wallet_private_key)
raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction

trx_hash = w3.eth.send_raw_transaction(raw_transaction)
print(f"Permit2 USTETH approve trx hash: {trx_hash.hex()}")

Checking the allowance we have just approved:

In [ ]:
print(f'Permit2 allowance for USTETH: {USTETH_contract.functions.allowance(wallet_public_address, permit2_address).call()}')

## Generating EIP-712 Signature for Permit2 (Off-chain)
This is a critical step. We need to generate a signature that authorizes the Universal Router to spend our USTETH via Permit2. This signature is generated off-chain and will be included in the transaction we send to the Universal Router.

In [ ]:
from uniswap_universal_router_decoder import FunctionRecipient, RouterCodec
from eth_account import Account
from eth_account.messages import encode_typed_data
import time

# prepare data for Permit2 signature and Universal Router call
PERMIT2_ADDRESS = "0x000000000022D473030F116dDEE9F6B43aC78BA3"
CHAIN_ID = 11155111 # Sepolia ID 

amount_in = int(0.1 * 10**18)
min_amount_out = 0

# Validity period for the signature (e.g., 30 days), used for Permit2 record-keeping
permit_expiration = int(time.time()) + (30 * 24 * 60 * 60)
# On-chain deadline for the signature (e.g., 20 minutes), the Router will reject the signature after this time
sig_deadline = int(time.time()) + (60 * 60)

# Get Permit2 Nonce (Critical Step)
# We must fetch the current nonce from the blockchain, otherwise the signature will be invalid
# allowance return structure: (amount, expiration, nonce)
try:
    _, _, current_nonce = permit2_contract.functions.allowance(
        wallet_public_address,      # Owner
        USTETH_token_address,       # Token
        universal_router_address    # Spender
    ).call()
except Exception as e:
    print(f"Error fetching nonce: {e}")
    # If this is the first interaction, the nonce might be 0
    current_nonce = 0

print(f"Current Nonce for Universal Router: {current_nonce}")

# Build and Generate EIP-712 Signature (Off-chain) 
# This is completely offline and gas-free
eip712_data = {
    "types": {
        "EIP712Domain": [
            {"name": "name", "type": "string"},
            {"name": "chainId", "type": "uint256"},
            {"name": "verifyingContract", "type": "address"},
        ],
        "PermitSingle": [
            {"name": "details", "type": "PermitDetails"},
            {"name": "spender", "type": "address"},
            {"name": "sigDeadline", "type": "uint256"},
        ],
        "PermitDetails": [
            {"name": "token", "type": "address"},
            {"name": "amount", "type": "uint160"},
            {"name": "expiration", "type": "uint48"},
            {"name": "nonce", "type": "uint48"},
        ],
    },
    "domain": {
        "name": "Permit2",
        "chainId": CHAIN_ID,
        "verifyingContract": PERMIT2_ADDRESS,
    },
    "primaryType": "PermitSingle",
    "message": {
        "details": {
            "token": USTETH_token_address,
            "amount": amount_in,        # Authorized amount, usually equal to or greater than the swap amount
            "expiration": permit_expiration,
            "nonce": current_nonce
        },
        "spender": universal_router_address, 
        "sigDeadline": sig_deadline,
    },
}

signed_message = Account.sign_typed_data(wallet_private_key, full_message=eip712_data)
print(f"Signature generated successfully: {signed_message}")

# Encode Universal Router instructions (Chained)
codec = RouterCodec()

# Here we chain permit2_permit and v3_swap_exact_in
# The Router will execute in order: first obtain authorization using the signature, then perform the swap
encoded_data = codec.encode.chain().permit2_permit(
    {
        "details": {
            "token": USTETH_token_address,
            "amount": amount_in,
            "expiration": permit_expiration,
            "nonce": current_nonce
        },
        "spender": universal_router_address,
        "sigDeadline": sig_deadline
    },
    signed_message
).v3_swap_exact_in(
    FunctionRecipient.SENDER,
    amount_in,
    min_amount_out,
    [   
        USTETH_token_address,
        100,
        USTUSD_token_address, 
    ],
).build(sig_deadline) 

print(f"Encoded data with Permit2: {encoded_data}")

### Checking the current Permit2 nonce, allowance, and expiration
For security purposes (prevent Double Spending), the message you need to sign contains a unique `nonce`. It is incremented for each permit message you sign. The nonce depends on your account address and on the token and universal router addresses.

We will find that although the allowance that permit2 approved to the universal router is 0, the universal router can still spend our USTETH because we have generated a valid signature that authorizes the universal router to spend our USTETH via permit2. This is the core idea of permit2. We do not need to approve the universal router directly, we just need to approve the permit2 contract and generate a valid signature.

In [ ]:
p2_amount, p2_expiration, p2_nonce = permit2_contract.functions.allowance(
        wallet_public_address,
        USTETH_token_address,
        universal_router_address
).call()

print(f'Permit2 allowance amount: {p2_amount}')
print(f'Permit2 allowance expiration: {p2_expiration}')
print(f'Permit2 allowance nonce: {p2_nonce}')


### Swap with Universal Router and Permit2

We now form swap transaction using universal router and v3_swap_exact_in:

First, we need to install Uniswap Universal Router Decoder & Encoder for encode and decode transaction params: 
```
pip install uniswap-universal-router-decoder
```
Then, you must **restart the kernel** to use the newly installed package.

Next, we encode the transaction input data. The input data is the part of the transaction that will be executed by the UR smart contract, resulting in the actual swap. We’ll use the UR codec as follow:

*Hint: If your `eth-account` package version is 0.10.0 or lower, you may upgrade it to 0.10.1 or higher to avoid compatibility issues with the UR codec. You can do this by running:*

`pip install --upgrade eth-account`

In [ ]:
from uniswap_universal_router_decoder import FunctionRecipient, RouterCodec
import time

deadline = int(time.time()) + (20 * 60) 

amount_in = int(0.1 * 10**18) # swap 0.1 USTETH into aprox 1980 USTUSD, in Wei (18 decimals for USTETH)
min_amount_out = 0

codec = RouterCodec() # initialize the codec, which will be used to encode the function calls to the Universal Router

encoded_data = codec.encode.chain().v3_swap_exact_in( # chain() serilizes the function calls; v3_swap_exact_in() encodes a Uniswap V3 swap with exact input amount, which is the function we want to call in this example
        FunctionRecipient.SENDER,  # reciver
        amount_in,  # swap amount in, in Wei
        min_amount_out,  # Slippage Protection: minimum amount out, in Wei. If below this, the transaction will revert
        [   
            USTETH_token_address, # swap-from token address ()
            100, # fee tier for Uniswap V3 pool (100 = 0.01%)
            USTUSD_token_address,  # swap-into token address, which is the token we want to receive
        ],
    ).build(deadline)  # unix timestamp after which the trx will not be valid any more (to prevent the melicious use of the transaction if it gets stuck in the mempool for a long time)

print(f"Encoded data for Universal Router execute function: {encoded_data}")

Some explanations
Let’s breakdown the command:


* encode : tells the codec we want to encode an input data (as opposed to decode).

* chain() : the UR supports several commands chained in a single transaction. chain() initialises the chaining for one or more sub-commands.

* v3_swap_exact_in() : Instruct the router to use a V3 pool with known input amount (1 ueth in this case).

* FunctionRecipient.SENDER : the transaction’s sender will receive the output

* min_amount_out: If the swap results in less that this amount of uhkd, the transaction will be reverted.

* codec.get_default_deadline() : the timestamp after which the transaction will not be valid anymore.

* build() : This method build and encode the transaction input data.

#### Execute the transaction (Swap: USTETH -> USTUSD)

Finally, we can execute the transaction by sending it to the Ethereum network. We will sign the transaction with our wallet’s private key and then send it.

In [ ]:
trx_params = {
        "from": wallet_public_address,
        "to": universal_router_address,
        "gas": 500_000,
        "maxPriorityFeePerGas": w3.eth.max_priority_fee,
        "maxFeePerGas": 100 * 10**9,
        "type": '0x2',
        "chainId": 11155111,
        "nonce": w3.eth.get_transaction_count(wallet_public_address),
        "data": encoded_data,
        }

signed_txn = w3.eth.account.sign_transaction(trx_params, wallet_private_key)
raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction
txn_hash = w3.eth.send_raw_transaction(raw_transaction)
print("Hash of universal router swap transaction : ", w3.to_hex(txn_hash))

- Check the changes in token balances on metamask